# 00 — Data Loader

Run this notebook first. It defines `preprocess_binary()` and `load_all_datasets()`
which are used by all other notebooks via `%run data_loader.ipynb`.

**All 8 CICIDS2017 files:**

| Key | File | Attack Types |
|---|---|---|
| `monday` | Monday-WorkingHours | Benign only |
| `bruteforce` | Tuesday-WorkingHours | FTP/SSH Brute Force |
| `dos` | Wednesday-workingHours | DoS attacks |
| `web_attacks` | Thursday-WorkingHours-Morning-WebAttacks | Brute Force, XSS, SQL Injection |
| `infiltration` | Thursday-WorkingHours-Afternoon-Infilteration | Infiltration |
| `botnet` | Friday-WorkingHours-Morning | Botnet |
| `portscan` | Friday-WorkingHours-Afternoon-PortScan | PortScan |
| `ddos` | Friday-WorkingHours-Afternoon-DDos | DDoS |

In [1]:
import pandas as pd
import numpy as np
import os

# ── Raw data directory ─────────────────────────────────────────────────────
RAW_DIR = "../data"

FILES = {
    "monday":       "Monday-WorkingHours.pcap_ISCX.csv",
    "bruteforce":   "Tuesday-WorkingHours.pcap_ISCX.csv",
    "dos":          "Wednesday-workingHours.pcap_ISCX.csv",
    "web_attacks": "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "infiltration":  "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "botnet":       "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "portscan":     "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "ddos":         "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
}
  
SOURCE_NAMES = {
    "monday":       "Monday",
    "bruteforce":   "BruteForce",
    "dos":          "DoS",
    "infiltration": "Infiltration",
    "web_attacks":  "WebAttacks",
    "botnet":       "Botnet",
    "portscan":     "PortScan",
    "ddos":         "DDoS",
}

print("Config loaded")

Config loaded


In [2]:
def preprocess_binary(df, source_name):
    """
    Clean a raw CICIDS2017 DataFrame and add binary label + source tag.
    
    Steps:
      1. Strip whitespace from column names
      2. Drop duplicate rows
      3. Drop rows with missing values
      4. Replace +/-inf with NaN, then drop again
      5. Add Label_Binary  (0 = BENIGN, 1 = attack)
      6. Add Source_File   (tracks which file each row came from)
    """
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.drop_duplicates()
    df = df.dropna()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()
    df["Label_Binary"] = (df["Label"] != "BENIGN").astype(int)
    df["Source_File"]  = source_name
    return df

print("preprocess_binary() defined")

preprocess_binary() defined


In [3]:
def load_all_datasets(keys=None):
    """
    Load and preprocess CICIDS2017 files.
    Returns a dict mapping key -> cleaned DataFrame.
    Missing files are skipped with a warning.

    Args:
        keys: list of keys to load, or None to load all 8.

    Example:
        datasets = load_all_datasets()
        df_web      = datasets["web_attacks"]
        df_portscan = datasets["portscan"]

        # Load only what you need
        datasets = load_all_datasets(["web_attacks", "portscan", "ddos"])
    """
    if keys is None:
        keys = list(FILES.keys())

    print(f"Loading {len(keys)} dataset(s): {keys}\n")

    datasets = {}
    for key in keys:
        path = os.path.join(RAW_DIR, FILES[key])
        print(f"  [{key}]  {FILES[key]}")
        try:
            raw = pd.read_csv(path, low_memory=False)
            df  = preprocess_binary(raw, SOURCE_NAMES[key])
            datasets[key] = df
            attack_rate = df["Label_Binary"].mean() * 100
            print(f"           raw {raw.shape}  ->  clean {df.shape}  "
                  f"(attack rate {attack_rate:.1f}%)")
        except FileNotFoundError:
            print(f"           SKIPPED - not found in {RAW_DIR}")

    print(f"\n  {len(datasets)}/{len(keys)} datasets loaded.")
    return datasets

print("load_all_datasets() defined")

load_all_datasets() defined


In [4]:
from pathlib import Path

def find_project_root():
    """Walk up from cwd until we find a folder containing both data/ and notebooks/."""
    current = Path.cwd().resolve()
    for folder in [current] + list(current.parents):
        if (folder / 'data').exists() and (folder / 'notebooks').exists():
            return folder
    return current

PROJECT_ROOT   = find_project_root()
CICIDS2018_DIR = PROJECT_ROOT / 'data' / 'cse_cic_ids2018'

print('Project root:         ', PROJECT_ROOT)
print('CICIDS2018 folder exists:', CICIDS2018_DIR.exists())

Project root:          /Users/yp_home/Documents/IDS_project
CICIDS2018 folder exists: True


In [5]:
def load_cicids2018(max_rows_per_file=30000):
    """
    Load CSE-CIC-IDS2018 CSV files from data/cse_cic_ids2018/.
    Applies the same cleaning as preprocess_binary().
    Returns a single cleaned DataFrame with Label_Binary (0/1) and Source_File columns.

    Args:
        max_rows_per_file: cap per CSV to avoid memory crashes (default 30000)

    Returns:
        df_2018: cleaned DataFrame ready for model training/testing
    """
    if not CICIDS2018_DIR.exists():
        print(f'WARNING: {CICIDS2018_DIR} not found. Download CSE-CIC-IDS2018 CSVs there.')
        return None

    csv_files = sorted(CICIDS2018_DIR.glob('*.csv'))
    if not csv_files:
        print('WARNING: No CSV files found in', CICIDS2018_DIR)
        return None

    print(f'Found {len(csv_files)} CSE-CIC-IDS2018 files')
    frames = []

    for file_path in csv_files:
        print(f'  Loading: {file_path.name}')
        try:
            df = pd.read_csv(file_path, low_memory=False, nrows=max_rows_per_file)
            df.columns = df.columns.str.strip()
            df['Source_File'] = file_path.stem
            frames.append(df)
        except Exception as e:
            print(f'  SKIPPED: {e}')

    if not frames:
        return None

    df_2018 = pd.concat(frames, ignore_index=True)

    # Clean
    df_2018.columns = df_2018.columns.str.strip()
    df_2018 = df_2018.drop_duplicates()
    df_2018 = df_2018.replace([np.inf, -np.inf], np.nan)
    df_2018 = df_2018.dropna()

    # Binary label — 2018 uses 'Benign' (mixed case)
    if 'Label' in df_2018.columns:
        df_2018['Label_Binary'] = (df_2018['Label'].str.strip().str.upper() != 'BENIGN').astype(int)
    else:
        print('WARNING: Label column not found')
        return df_2018

    attack_rate = df_2018['Label_Binary'].mean() * 100
    print(f'\n2018 clean shape: {df_2018.shape}  attack rate: {attack_rate:.1f}%')
    print('Label distribution:')
    print(df_2018['Label'].value_counts().head(10))
    return df_2018

print('load_cicids2018() defined')

load_cicids2018() defined


## How to use in other notebooks

Add this at the top of any notebook:

```python
%run data_loader.ipynb

# Load CICIDS2017 files
datasets = load_all_datasets(["web_attacks", "portscan", "ddos"])
df_web = datasets["web_attacks"]

# Load CICIDS2018 (put CSVs in data/cse_cic_ids2018/)
df_2018 = load_cicids2018(max_rows_per_file=30000)
```